---
title: Autograd
numbering: false
---

In [1]:
# Environment Setup
import torch
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # cuda가 가능한 환경의 경우 cuda로
print(f"사용하고 있는 device: {device}")

사용하고 있는 device: mps


# 0. 개요

PyTorch는 연산을 수행할 때마다 무엇을 가지고 무엇을 만들었는지를 기록해둔다. 이러한 기록을 computational graph라 한다. `backward()`를 이용하면 이 그래프를 거꾸로 따라가며 chain rule을 이용하여 gradient를 계산한다. 즉, autograd, 자동미분이다.(우리가 따로 미분연산을 할 필요가 없는 이유)

# 1. requires_grad, grad_fn, leaf

In [2]:
x = torch.tensor(2., requires_grad=True)
y = x**2 + 3*x # y = x^2 + 3x
print(y) # tensor(10., grad_fn=<AddBackward0>)
print(x.is_leaf, y.is_leaf)

tensor(10., grad_fn=<AddBackward0>)
True False


+ grad_fn: 이 텐서를 만든 연산에 대한 기록이다. `y`는 덧셈(<AddBackward0>)으로 만들어졌다고 적혀 있음
+ leaf: 연산에 대한 결과가 아니라 사용자가 직접 만든 텐서라는 의미.

In [3]:
y.backward() # dy/dx
print(x.grad) # 2x + 3 = 7 (7.,)

tensor(7.)


# 2. .grad는 덮어쓰는게 아니라 누적이 된다. (accumalative)

In [4]:
x = torch.tensor(2., requires_grad=True)
for _ in range(2):
    y = x**2 + 3*x
    y.backward()
    print(x.grad)

tensor(7.)
tensor(14.)


+ 두 번째 backward()에서 7이 새로 들어가는 것이 아닌 기존 7에 새로운 7이 더해져서 14가 된다.
    + 학습 루프를 짤 때 optim.zero_grad()를 매 스텝 호출하는 이유가 이 때문이다.
    + 같은 그래프에 backward()를 두 번 호출하면 runtime error 가 발생하는데 메모리를 아끼기 위해서 한 번 backward()가 끝나면 그래프를 지우기 때문

# 3. backward는 스칼라 값에서 시작함

In [5]:
v = torch.tensor([1., 2., 3.], requires_grad=True)
out = v * 2
# out.backward() # RuntimeError: grad can be implicitly created only for scalar outputs
out.sum().backward() # dy/dx = 2

In [6]:
print(v.grad)

tensor([2., 2., 2.])


+ 손실함수가 항상 .mean() 이나 .sum()으로 끝나서 스칼라 값을 내놓는 이유이기도 함

# 4. 기록을 끄는 방법: `torch_no_grad()`

In [7]:
w = torch.tensor(3., requires_grad=True)
with torch.no_grad():
    z = w * 2
print(z.requires_grad, z.grad_fn)

False None


+ 그래프를 아예 만들지 않기 때문에 평가나 추론처럼 미분이 필요한 과정이 아닐 경우 메모리와 시간을 아낄 수 있다.

# 5. 경로 짜르기: `.detach()`
+ stop gradient 등의 방법에서 사용

In [8]:
x = torch.tensor(3., requires_grad=True)

z = x * x
z.backward() # dy/dx = 2x
print(x.grad) # (6.,)

tensor(6.)


In [9]:
x.grad = None
z = x * x.detach()
z.backward()
print(x.grad) # (3.,) 왜 절반의 값이 나올까?

tensor(3.)


- x.detach()는 값의 경우 x와 같지만 그래프에서는 떨어진 텐서이다. 따라서 미분할 때 상수 취급이 된다.
    + no_grad와 detach의 차이
        + no_grad: 블록 전체에서 기록을 하지 않는다
        + detach: 특정 텐서 하나만 그래프에서 떼어 낸다.
    + 이를 이용하여 어느 방향으로 그래디언트를 흘려보낼지를 결정할 수 있음

# 6. 미분 불가능 연산과 straight through trick

In [10]:
x = torch.tensor(2.7, requires_grad=True)
r = x.round()
r.backward()
print(r.item(), x.grad) # 그래디언트가 0이 나온다

3.0 tensor(0.)


+ 반올림이나 argmin처럼 값을 고르는 연산은 그래디언트가 0이거나 없다. 이를 해결하기 위해서 쓰는 trick

In [11]:
x.grad = None # 그래디언트가 없다고 설정
y = x + (x.round() - x).detach()
y.backward()
print(y.item(), x.grad)

3.0 tensor(1.)


+ forward: x - x + x.round() 가 계산되므로 그대로 반올림값된 값이 지나간다.
+ backward: 괄호 부분이 detach() 되어 상수취급 되므로 $y = x + \text{상수}$ 가 되고 gradient 는 1이 된다.

# 7. 그래디언트를 직접 꺼내서 확인하기: `torch.autograd.grad`

In [12]:
x = torch.tensor(2., requires_grad=True)
y = x**3
(g,) = torch.autograd.grad(y, x) # .grad 안에 누적하는게 아니라 그래디언트 값만 가져오게 됨
print(g, x.grad) # 따라서 x.grad 에는 아무것도 없고 g로 확인 가능

tensor(12.) None
